# 01 - Classification: Opportunity Result

Pipeline:
1. OptBinning transformer
2. LogisticRegression

Data source: `../../data/intermediate/df_train_stratified.parquet`.

In [117]:
import numpy as np
import pandas as pd

from sklearn.base import clone
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, RobustScaler, OneHotEncoder
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score, average_precision_score, accuracy_score, f1_score
from sklearn.linear_model import LogisticRegression

from xgboost import XGBClassifier

from binning import NamedBinningProcess, DataFrameScaler
from statsmodels_api import StatsModelsClassifier

import statsmodels.api as sm
from tqdm.auto import tqdm




pd.set_option('display.max_columns', 200)

In [3]:
df = pd.read_parquet('../../data/intermediate/df_train_stratified.parquet')
print('shape:', df.shape)

shape: (53073, 37)


In [85]:
# Editable feature list (all enabled by default)
FEATURES = [
    'Supplies Group',
    'Supplies Subgroup',
    'Region',
    'Route To Market',
    'Elapsed Days In Sales Stage',
    'Sales Stage Change Count',
    'Total Days Identified Through Closing',
    'Total Days Identified Through Qualified',
    #'Opportunity Amount USD',
    'Client Size By Revenue (USD)',
    'Client Size By Employee Count',
    'Revenue From Client Past Two Years (USD)',
    'Competitor Type',
    'Ratio Days Identified To Total Days',
    'Ratio Days Validated To Total Days',
    'Ratio Days Qualified To Total Days',
    #'Deal Size Category (USD)',
    #'total_days_zero',
    #'opportunity_amount_weirdness',
    #'flag_0_days',
    #'flag_ratio_problem',
    #'flag_zero_opportunity_amount',
    #'flag_outlier_opportunity_amount',
    #'flag_outlier_total_days',
    #'flag_totally_repeated_row',
    #'flag_partially_repeated_row',
    #'partial_repeat_is_latest_id_appearance',
    #'flag_only_identified',
    #'flag_weirdness_over_75th_pct',
    #'problem_count',
]

TARGET = 'Opportunity Result Bool'

missing = [c for c in FEATURES + [TARGET] if c not in df.columns]
if missing:
    raise ValueError(f'Missing columns: {missing}')

X = df[FEATURES].copy()
y = df[TARGET].astype(int).copy()
print('n_features:', len(FEATURES))
print('target mean (Won rate):', y.mean())

n_features: 15
target mean (Won rate): 0.22725302884705972


In [86]:
#X["flag_0_days"] = (X["Total Days Identified Through Closing"] == 0).astype(str)
#X["flag_ratio_problem"] = (X["Total Days Identified Through Closing"] == 0).astype(str)


In [94]:
X_cols = list(X.columns)
X_categorical_cols = [
    'Supplies Group',
    'Supplies Subgroup',
    'Region',
    'Route To Market',
    'Competitor Type',
    'Client Size By Revenue (USD)',
    'Client Size By Employee Count',
    'Revenue From Client Past Two Years (USD)',
#    'flag_0_days',
#    'flag_ratio_problem',
]
X_numerical_cols = [c for c in X_cols if c not in X_categorical_cols]

In [95]:
def get_classic_preprocessor(scaler=StandardScaler(), imputer_strategy_numerical="median", imputer_strategy_categorical="most_frequent", categorical_cols=X_categorical_cols, numerical_cols=X_numerical_cols):
    return ColumnTransformer([
        ('numerical', Pipeline([
            ("imputer", SimpleImputer(strategy=imputer_strategy_numerical)),
            ("scaler", scaler)
        ]), numerical_cols),
        ('categorical', Pipeline([
            ("imputer", SimpleImputer(strategy="constant", fill_value="MISSING")),
            ("ohe", OneHotEncoder(drop="first", handle_unknown='ignore', sparse_output=False))
        ]), categorical_cols)
    ]).set_output(transform="pandas")

In [ ]:
experiment_grid = {
    "classic_logit_standard_scaler": Pipeline([
        ("preprocessing",get_classic_preprocessor(scaler=StandardScaler(), imputer_strategy_numerical="median", imputer_strategy_categorical="most_frequent", categorical_cols=X_categorical_cols, numerical_cols=X_numerical_cols)),
        ('model', StatsModelsClassifier(sm.Logit))
    ]),
    "classic_probit_standard_scaler": Pipeline([
        ("preprocessing", get_classic_preprocessor(scaler=StandardScaler(), imputer_strategy_numerical="median", imputer_strategy_categorical="most_frequent", categorical_cols=X_categorical_cols, numerical_cols=X_numerical_cols)),
        ('model', StatsModelsClassifier(sm.Probit))
    ]),
    "classic_logit_robust_scaler": Pipeline([
        ("preprocessing", get_classic_preprocessor(scaler=RobustScaler(), imputer_strategy_numerical="median", imputer_strategy_categorical="most_frequent", categorical_cols=X_categorical_cols, numerical_cols=X_numerical_cols)),
        ('model', StatsModelsClassifier(sm.Logit))
    ]),
    "classic_probit_robust_scaler": Pipeline([
        ("preprocessing", get_classic_preprocessor(scaler=RobustScaler(), imputer_strategy_numerical="median", imputer_strategy_categorical="most_frequent", categorical_cols=X_categorical_cols, numerical_cols=X_numerical_cols)),
        ('model', StatsModelsClassifier(sm.Probit))
    ]),
    "logit_elasticnet_robust_scaler": Pipeline([
        ("preprocessing", get_classic_preprocessor(scaler=RobustScaler(), imputer_strategy_numerical="median", imputer_strategy_categorical="most_frequent", categorical_cols=X_categorical_cols, numerical_cols=X_numerical_cols)),
        ('model', LogisticRegression(solver='saga', l1_ratio=0.5, max_iter=10000))
    ]),
    "logit_elasticnet_standard_scaler": Pipeline([
        ("preprocessing", get_classic_preprocessor(scaler=StandardScaler(), imputer_strategy_numerical="median", imputer_strategy_categorical="most_frequent", categorical_cols=X_categorical_cols, numerical_cols=X_numerical_cols)),
        ('model', LogisticRegression(solver='saga', l1_ratio=0.5, max_iter=10000))
    ]),
    "probit_binned_ohe": Pipeline([
        ('binning', NamedBinningProcess(
            variable_names=X_cols,
            categorical_variables=X_categorical_cols,
            output_method="bin_ohe"
        )),
        ('model', StatsModelsClassifier(sm.Probit))
    ]),
    "logit_binned_ohe": Pipeline([
        ('binning', NamedBinningProcess(
            variable_names=X_cols,
            categorical_variables=X_categorical_cols,
            output_method="bin_ohe"
        )),
        ('model', StatsModelsClassifier(sm.Logit))
    ]),
    "logit_elasticnet_binned_ohe": Pipeline([
        ('binning', NamedBinningProcess(
            variable_names=X_cols,
            categorical_variables=X_categorical_cols,
            output_method="bin_ohe"
        )),
        ('model', LogisticRegression(solver='saga', l1_ratio=0.5, max_iter=10000))
    ]),
    "probit_binned_woe_robust_scaler": Pipeline([
        ('binning', NamedBinningProcess(
            variable_names=X_cols,
            categorical_variables=X_categorical_cols,
            output_method="woe"
        )),
        ('scaler', DataFrameScaler(RobustScaler())),
        ('model', StatsModelsClassifier(sm.Probit))
    ]),
    "logit_binned_woe_robust_scaler": Pipeline([
        ('binning', NamedBinningProcess(
            variable_names=X_cols,
            categorical_variables=X_categorical_cols,
            output_method="woe"
        )),
        ('scaler', DataFrameScaler(RobustScaler())),
        ('model', StatsModelsClassifier(sm.Logit))
    ]),
    "probit_binned_woe": Pipeline([
        ('binning', NamedBinningProcess(
            variable_names=X_cols,
            categorical_variables=X_categorical_cols,
            output_method="woe"
        )),
        #('scaler', DataFrameScaler(RobustScaler())),
        ('model', StatsModelsClassifier(sm.Probit))
    ]),
    "logit_binned_woe": Pipeline([
        ('binning', NamedBinningProcess(
            variable_names=X_cols,
            categorical_variables=X_categorical_cols,
            output_method="woe"
        )),
        #('scaler', DataFrameScaler(RobustScaler())),
        ('model', StatsModelsClassifier(sm.Logit))
    ]),
    "logit_elastic_binned_woe": Pipeline([
        ('binning', NamedBinningProcess(
            variable_names=X_cols,
            categorical_variables=X_categorical_cols,
            output_method="woe"
        )),
        #('scaler', DataFrameScaler(RobustScaler())),
        ('model', LogisticRegression(solver='saga', l1_ratio=0.5, max_iter=10000))
    ]),
    "xgboost": Pipeline([
        ('preprocessing', get_classic_preprocessor(scaler=None, imputer_strategy_numerical="median", imputer_strategy_categorical="most_frequent", categorical_cols=X_categorical_cols, numerical_cols=X_numerical_cols)),
        ('model', XGBClassifier(use_label_encoder=False, eval_metric='logloss'))
    ])
}

In [124]:
cv = 5
skf = StratifiedKFold(n_splits=cv, shuffle=True, random_state=42)

results = []

In [125]:
outer_bar = tqdm(experiment_grid.items(), total=len(experiment_grid), desc="Experiments", position=0)

for exp_name, pipeline in outer_bar:
    outer_bar.set_postfix_str(exp_name)

    inner_bar = tqdm(
        enumerate(skf.split(X, y), start=1),
        total=cv,
        desc=f"{exp_name} folds",
        leave=False,
        position=1
    )

    for fold, (train_idx, valid_idx) in inner_bar:
        X_train = X.iloc[train_idx] if hasattr(X, "iloc") else X[train_idx]
        X_valid = X.iloc[valid_idx] if hasattr(X, "iloc") else X[valid_idx]

        y_train = y.iloc[train_idx] if hasattr(y, "iloc") else y[train_idx]
        y_valid = y.iloc[valid_idx] if hasattr(y, "iloc") else y[valid_idx]

        model = clone(pipeline)

        try:
            model.fit(X_train, y_train)

            # adjust if your wrapper does not support predict_proba
            y_proba = model.predict_proba(X_valid)[:, 1]
            y_pred = (y_proba >= 0.5).astype(int)

            results.append({
                "experiment": exp_name,
                "fold": fold,
                "roc_auc": roc_auc_score(y_valid, y_proba),
                "pr_auc": average_precision_score(y_valid, y_proba),
                "accuracy": accuracy_score(y_valid, y_pred),
                "f1": f1_score(y_valid, y_pred, zero_division=0),
                "status": "ok",
                "error": None,
            })

        except Exception as e:
            results.append({
                "experiment": exp_name,
                "fold": fold,
                "roc_auc": np.nan,
                "pr_auc": np.nan,
                "accuracy": np.nan,
                "f1": np.nan,
                "status": "failed",
                "error": str(e),
            })

Experiments:   0%|          | 0/15 [00:00<?, ?it/s, classic_logit_standard_scaler]

Optimization terminated successfully.
         Current function value: 0.364942
         Iterations 8


Optimization terminated successfully.
         Current function value: 0.361723
         Iterations 7


Optimization terminated successfully.
         Current function value: 0.365052
         Iterations 7


Optimization terminated successfully.
         Current function value: 0.362700
         Iterations 7


...

Optimization terminated successfully.
         Current function value: 0.352497
         Iterations 7


Optimization terminated successfully.
         Current function value: 0.350224
         Iterations 7


Experiments:  87%|████████▋ | 13/15 [01:04<00:08,  4.29s/it, logit_elastic_binned_woe]

Optimization terminated successfully.
         Current function value: 0.352217
         Iterations 7


/Users/goviedb/Development/mib_cars_analysis/.venv/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
/Users/goviedb/Development/mib_cars_analysis/.venv/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
/Users/goviedb/Development/mib_cars_analysis/.venv/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1135: FutureWarnin

In [126]:
results_df = pd.DataFrame(results)

summary_df = (
    results_df
    .groupby("experiment", dropna=False)
    .agg(
        roc_auc_mean=("roc_auc", "mean"),
        roc_auc_std=("roc_auc", "std"),
        pr_auc_mean=("pr_auc", "mean"),
        pr_auc_std=("pr_auc", "std"),
        accuracy_mean=("accuracy", "mean"),
        accuracy_std=("accuracy", "std"),
        f1_mean=("f1", "mean"),
        f1_std=("f1", "std"),
        n_failed=("status", lambda s: (s == "failed").sum()),
    )
    .sort_values("roc_auc_mean", ascending=False)
    .reset_index()
)

results_df, summary_df

(                       experiment  fold   roc_auc    pr_auc  accuracy  \
 0   classic_logit_standard_scaler     1  0.871232  0.680131  0.829016   
 1   classic_logit_standard_scaler     2  0.858954  0.664845  0.826566   
 2   classic_logit_standard_scaler     3  0.871635  0.685353  0.831371   
 3   classic_logit_standard_scaler     4  0.864924  0.660839  0.826644   
 4   classic_logit_standard_scaler     5  0.870835  0.681718  0.828811   
 ..                            ...   ...       ...       ...       ...   
 70                        xgboost     1  0.924998  0.814766  0.883749   
 71                        xgboost     2  0.921457  0.808627  0.876213   
 72                        xgboost     3  0.930026  0.820784  0.886858   
 73                        xgboost     4  0.923584  0.810998  0.878933   
 74                        xgboost     5  0.927840  0.816654  0.881854   
 
           f1 status error  
 0   0.547269     ok  None  
 1   0.541012     ok  None  
 2   0.555611     ok  N

In [127]:
summary_df

,experiment,roc_auc_mean,roc_auc_std,pr_auc_mean,pr_auc_std,accuracy_mean,accuracy_std,f1_mean,f1_std,n_failed
0,xgboost,0.925581,0.003398,0.814365,0.004766,0.881522,0.004136,0.718448,0.010749,0
1,probit_binned_ohe,0.881099,0.004329,0.699451,0.005430,0.840842,0.002957,0.588763,0.009848,0
2,logit_binned_ohe,0.881053,0.004400,0.700006,0.005493,0.841935,0.003374,0.595983,0.010938,0
3,logit_elasticnet_binned_ohe,0.881040,0.004398,0.700107,0.005481,0.841878,0.003073,0.594344,0.009979,0
4,probit_binned_woe,0.877249,0.003758,0.692510,0.003675,0.837130,0.002506,0.563799,0.009740,0
5,probit_binned_woe_robust_scaler,0.877249,0.003758,0.692510,0.003675,0.837130,0.002506,0.563799,0.009740,0
6,logit_elastic_binned_woe,0.877158,0.003848,0.693005,0.003656,0.837771,0.002415,0.570011,0.008637,0
7,logit_binned_woe,0.877151,0.003837,0.693000,0.003681,0.837620,0.002470,0.569616,0.008344,0
8,logit_binned_woe_robust_scaler,0.877151,0.003837,0.693000,0.003681,0.837620,0.002470,0.569616,0.008344,0
9,logit_elasticnet_standard_scaler,0.867528,0.005530,0.674541,0.010971,0.828425,0.002082,0.546419,0.008390,0
